In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5


@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [4]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
# create subagents
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
subagent_1 = create_agent(
    model=model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model=model,
    tools=[square]
)

## Calling subagents

In [6]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [7]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [8]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='a276c1a1-c113-430b-90d5-41bf5158265a'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'call_subagent_1', 'arguments': '{"x": 456}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cf267-ff8f-7ee2-b117-43d8e56512b1-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'a3ffb31b-dd01-4f37-ab1d-92def649d1bd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 132, 'output_tokens': 20, 'total_tokens': 152, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content='The square root of 456.0 is 21.354156504062622.', name='call_subagent_1', id='1bfdb89f-a794-4c2c-a15f-dcdac8b367de', tool_call_id='a3ffb31b-dd01-4f37-ab1d-92def649d1bd'),
              AIMessage(content='The sq